# 🩻 Foundation Models para Radiologia: ImageNet vs. Domínio Clínico
## Hands-on Vibe Coding — Sociedade Paulista de Radiologia 2026

---

### A Pergunta Central

Um modelo treinado em fotos do cotidiano (gatos, carros, 1.000 classes) tem o mesmo valor como extrator de características radiológicas que um modelo treinado em **880 mil raios-X de tórax**?

Neste hands-on você vai responder isso experimentalmente:

| Extrator | Pré-treinamento | O que aprendeu |
|---|---|---|
| **EfficientNetB0** | ImageNet (1,28M fotos naturais) | Bordas, texturas, formas gerais |
| **RAD-DINO** | 880K+ raios-X de tórax (Microsoft, 2024) | Opacidades, consolidações, textura pulmonar |

**Experimento:** congele ambos os backbones → treine a **mesma cabeça classificadora** em cada → compare o t-SNE dos embeddings e as métricas no conjunto de teste.

---

### Dataset: PneumoniaMNIST

- **1.000 imagens** de raio-X de tórax (64×64 pixels, subsample reproduzível)
- **2 classes:** Normal vs Pneumonia Bacteriana/Viral
- Download automático via `pip install medmnist` — sem login, sem cadastro
- Fonte: Kermany et al., *Cell* 2018 | Licença: CC BY 4.0

---

### Como usar este notebook

1. **Ative o GPU T4:** Menu `Ambiente de execução` → `Alterar tipo de hardware` → T4 GPU → Salvar → Reconectar
2. **Execute as células de Setup, Dataset e Embeddings** — elas já estão pré-preenchidas e preparam todo o ambiente
3. A partir do **PC1**, use o **Gemini** (ícone ✦ ou `Ctrl+Shift+I`): selecione a célula vazia, cole o prompt do card projetado no Gemini e clique em **Inserir** para colocar o código diretamente na célula

> 💡 **Setup / Dataset / Embeddings:** Execute diretamente (código pronto)  
> 🤖 **PC1–PC4:** Construa via Gemini (selecione a célula vazia → cole o prompt → Inserir → Execute)

---

### ⏱️ Cronograma (75 minutos)

| | Bloco | Tempo | Modo |
|---|---|---|---|
| 🔧 | Setup e Preparação | 5 min | ▶ Execute |
| 📦 | Carregar e Visualizar o Dataset | 5 min | ▶ Execute |
| 🎬 | Extração de Embeddings + t-SNE | 10 min | ▶ Execute |
| 🏗️ | Classificador 1: EfficientNetB0 (ImageNet) | 12 min | ✦ PC1 Gemini |
| 🏗️ | Classificador 2: RAD-DINO (Radiologia) | 12 min | ✦ PC2 Gemini |
| 📊 | Comparação de Resultados | 8 min | ✦ PC3 Gemini |
| 🔎 | Inferência em Imagem Própria | 10 min | ✦ PC4 Gemini |
| 💬 | Discussão Clínica | 13 min | — |


## 🔧 Setup e Preparação
### Prompt Card 0 — ▶ Execute a célula abaixo


In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Ignorar avisos do medmnist para deixar o output mais limpo

# ══════════════════════════════════════════════════════════════════════════════
# INSTALAÇÕES
# Instalamos as bibliotecas necessárias para o experimento:
#   - medmnist      → dataset de imagens médicas (inclui PneumoniaMNIST)
#   - torchvision   → modelos pré-treinados de visão (EfficientNetB0)
#   - transformers  → modelos de linguagem e visão da Hugging Face (RAD-DINO)
#   - accelerate    → suporte para treinamento eficiente com GPU
#   - scikit-learn  → métricas e t-SNE
#   - matplotlib/seaborn → visualizações
# ══════════════════════════════════════════════════════════════════════════════
!pip install -qqq medmnist torchvision transformers accelerate scikit-learn matplotlib seaborn

# ── Imports ───────────────────────────────────────────────────────────────────
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as transforms
import torchvision

from medmnist import PneumoniaMNIST

from transformers import AutoImageProcessor, AutoModel  # Para carregar o RAD-DINO

from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.manifold import TSNE

# ══════════════════════════════════════════════════════════════════════════════
# SEMENTE DE ALEATORIEDADE (SEED)
# SEED = 42 garante que todo resultado aleatório seja reproduzível:
# embaralhamento de dados, inicialização de pesos, subsampling, etc.
# Qualquer pessoa que rodar este notebook vai obter exatamente os mesmos resultados.
# ══════════════════════════════════════════════════════════════════════════════
SEED = 42

def set_seed(seed):
    """Fixa a semente em todas as bibliotecas de aleatoriedade usadas no experimento."""
    random.seed(seed)           # Python nativo
    np.random.seed(seed)        # NumPy
    torch.manual_seed(seed)     # PyTorch (CPU)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)      # PyTorch (GPU única)
        torch.cuda.manual_seed_all(seed)  # PyTorch (múltiplas GPUs)
    torch.backends.cudnn.deterministic = True  # Operações CUDA determinísticas
    torch.backends.cudnn.benchmark = False     # Desativa otimizações não-determinísticas

set_seed(SEED)

# ══════════════════════════════════════════════════════════════════════════════
# DEVICE
# Verifica se há GPU disponível no Colab e define o "device" de treinamento.
# Se o GPU T4 estiver ativo, DEVICE = 'cuda'; caso contrário, 'cpu'.
# Todas as operações tensoriais serão enviadas para este device.
# ══════════════════════════════════════════════════════════════════════════════
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando device: {DEVICE}')

# ══════════════════════════════════════════════════════════════════════════════
# FUNÇÕES AUXILIARES
# Definimos aqui as funções que serão reutilizadas nas etapas seguintes.
# ══════════════════════════════════════════════════════════════════════════════

def numpy_to_pil(images_np):
    """
    Converte um array numpy de imagens (N, H, W, C) para uma lista de imagens PIL.

    O RAD-DINO espera receber imagens no formato PIL — não tensores.
    Esta função re-escala os valores de [0,1] para [0,255] e converte cada imagem.
    Se a imagem for grayscale (1 canal), ela é convertida para PIL em modo 'L';
    caso seja RGB (3 canais), é convertida normalmente.
    """
    images_np = (images_np * 255).astype(np.uint8)
    pil_images = [Image.fromarray(img[:,:,0] if img.shape[-1] == 1 else img) for img in images_np]
    return pil_images

def extrair_embeddings_cnn(model, images_np, batch_size=32):
    """
    Extrai embeddings de um modelo CNN (ex: EfficientNetB0).

    Processo:
    1. Aplica a normalização ImageNet (média e desvio padrão do ImageNet)
       porque o EfficientNetB0 foi treinado com essas estatísticas.
    2. Organiza as imagens em mini-batches (lotes de 32) para não sobrecarregar a GPU.
    3. Roda o modelo em modo de avaliação (sem dropout, sem gradientes).
    4. Empilha todos os embeddings em um único array numpy.

    Retorna: array numpy de shape (N, D), onde D é a dimensão do embedding (1280 para EfficientNetB0).
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = TensorDataset(torch.stack([transform(Image.fromarray((img * 255).astype(np.uint8))) for img in images_np]))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    embeddings = []
    model.eval()
    with torch.no_grad():  # Desativa cálculo de gradientes — economiza memória e acelera
        for batch in dataloader:
            img_batch = batch[0].to(DEVICE)
            emb = model(img_batch).cpu().numpy()
            embeddings.append(emb)
    return np.vstack(embeddings)

def extrair_embeddings_raddino(model, processor, images_pil, batch_size=32):
    """
    Extrai embeddings do modelo RAD-DINO.

    O RAD-DINO é um Vision Transformer (ViT) treinado com DINO em 880K+ raios-X.
    Ao contrário de CNNs, ele usa atenção global — cada token vê toda a imagem.
    O embedding que usamos é o CLS token: o vetor de 768 dimensões que o modelo
    produz como "resumo" de toda a imagem, análogo ao embedding de [CLS] em BERT.

    O 'processor' cuida do pré-processamento (resize, normalização) no padrão do RAD-DINO.
    """
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(images_pil), batch_size):
            batch_pil = images_pil[i:i+batch_size]
            inputs = processor(images=batch_pil, return_tensors='pt').to(DEVICE)
            outputs = model(**inputs)
            # last_hidden_state[:, 0, :] → CLS token (posição 0 da sequência de tokens)
            emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(emb)
    return np.vstack(embeddings)

# ══════════════════════════════════════════════════════════════════════════════
# ARQUITETURA DA CABEÇA CLASSIFICADORA
# Esta é a rede neural que será treinada em cima dos embeddings congelados.
# Ela é propositalmente simples — o objetivo é testar a qualidade do extrator,
# não fazer engenharia de features.
#
# Arquitetura: Linear(D→256) → ReLU → Dropout(0.3) → Linear(256→2)
#   - D: dimensão do embedding de entrada (1280 para EfficientNetB0, 768 para RAD-DINO)
#   - 256: camada oculta intermediária
#   - ReLU: ativação não-linear
#   - Dropout(0.3): regularização — durante treino, 30% dos neurônios são zerados aleatoriamente
#   - Linear(256→2): 2 saídas → Normal e Pneumonia (logits para CrossEntropyLoss)
# ══════════════════════════════════════════════════════════════════════════════
class ClassificationHead(nn.Module):
    def __init__(self, in_features, num_classes=2, dropout_rate=0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.head(x)

def treinar_cabeca(emb_train, y_train, emb_val, y_val, in_features, epochs=50, lr=1e-3, batch_size=32):
    """
    Treina a cabeça classificadora com os embeddings fornecidos.

    Parâmetros de treinamento (iguais para os dois modelos — fair comparison):
      - Otimizador: Adam com lr=1e-3 (adaptativo, converge rápido)
      - Função de perda: CrossEntropyLoss (padrão para classificação multi-classe)
      - Epochs: 50 passagens completas pelo conjunto de treino
      - Batch size: 32 imagens por atualização de gradiente

    A cada época, calcula o AUC na validação para monitorar overfitting.
    Retorna o modelo treinado e o histórico de loss/AUC por época.
    """
    set_seed(SEED)  # Garante que os pesos iniciais são os mesmos para os dois modelos

    head = ClassificationHead(in_features).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_dataset = TensorDataset(torch.tensor(emb_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = TensorDataset(torch.tensor(emb_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    history = {'loss': [], 'val_auc': []}

    for epoch in range(epochs):
        # ── Fase de Treino ──────────────────────────────────────────────────
        head.train()  # Ativa dropout e outras camadas de treino
        total_loss = 0
        for embeddings, labels in train_loader:
            embeddings, labels = embeddings.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()       # Zera gradientes do passo anterior
            outputs = head(embeddings)  # Forward pass
            loss = criterion(outputs, labels)
            loss.backward()             # Backpropagation — calcula gradientes
            optimizer.step()            # Atualiza pesos
            total_loss += loss.item()
        history['loss'].append(total_loss / len(train_loader))

        # ── Fase de Validação ───────────────────────────────────────────────
        head.eval()  # Desativa dropout para avaliação
        val_preds, val_true, val_probs = [], [], []
        with torch.no_grad():
            for embeddings, labels in val_loader:
                embeddings, labels = embeddings.to(DEVICE), labels.to(DEVICE)
                outputs = head(embeddings)
                val_probs.extend(torch.softmax(outputs, dim=1)[:, 1].cpu().numpy())
                val_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
                val_true.extend(labels.cpu().numpy())
        val_auc = roc_auc_score(val_true, val_probs)
        history['val_auc'].append(val_auc)

    return head, history

def avaliar(head, emb_test, y_test, model_name):
    """
    Avalia o modelo treinado no conjunto de teste e imprime as métricas clínicas.

    Métricas retornadas:
      - Acurácia: % de predições corretas no total
      - AUC: área sob a curva ROC (quanto melhor o ranking entre classes)
      - Precisão (VPP): dos que predisse como Pneumonia, quantos de fato têm?
      - Recall (Sensibilidade): dos que têm Pneumonia, quantos o modelo detectou?
      - F1-Score: média harmônica entre precisão e recall
      - Matriz de Confusão: tabela TP/FP/FN/TN

    Retorna um dicionário com todas as métricas para uso posterior (comparação).
    """
    head.eval()
    with torch.no_grad():
        embeddings_tensor = torch.tensor(emb_test, dtype=torch.float32).to(DEVICE)
        outputs = head(embeddings_tensor)
        test_probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        test_preds = torch.argmax(outputs, dim=1).cpu().numpy()

    accuracy  = accuracy_score(y_test, test_preds)
    auc       = roc_auc_score(y_test, test_probs)
    precision = precision_score(y_test, test_preds)
    recall    = recall_score(y_test, test_preds)
    f1        = f1_score(y_test, test_preds)
    cm        = confusion_matrix(y_test, test_preds)

    print(f'\n--- Avaliação: {model_name} ---')
    print(f'Acurácia: {accuracy:.4f}')
    print(f'AUC: {auc:.4f}')
    print(f'Precisão (VPP): {precision:.4f}')
    print(f'Recall (Sensibilidade): {recall:.4f}')
    print(f'F1-Score: {f1:.4f}')
    print(f'Matriz de Confusão:\n{cm}')

    return {'model_name': model_name, 'accuracy': accuracy, 'auc': auc, 'precision': precision, 'recall': recall, 'f1': f1}


## 📦 Carregar e Visualizar o Dataset
### Prompt Card 1 — ▶ Execute a célula abaixo


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CARREGAMENTO DO DATASET: PneumoniaMNIST
# O medmnist baixa automaticamente o dataset (sem necessidade de login).
# PneumoniaMNIST: raios-X de tórax em 64×64 pixels, 2 classes:
#   0 = Normal  |  1 = Pneumonia (bacteriana ou viral)
# Vem dividido em treino / validação / teste pelo próprio pacote.
# ══════════════════════════════════════════════════════════════════════════════
train_ds = PneumoniaMNIST(split='train', download=True, size=64)
val_ds   = PneumoniaMNIST(split='val',   download=True, size=64)
test_ds  = PneumoniaMNIST(split='test',  download=True, size=64)

def preprocess(ds):
    """
    Pré-processa um split do PneumoniaMNIST:
      1. Converte pixels de uint8 [0,255] para float32 [0,1]  → normalização básica
      2. Garante que as imagens tenham 4 dimensões (N, H, W, C)
      3. Replica o canal grayscale 3 vezes → (N, H, W, 3)
         Isso é necessário porque EfficientNetB0 e RAD-DINO esperam 3 canais RGB
      4. Achata as labels para um array 1D de inteiros
    """
    imgs = ds.imgs.astype('float32') / 255.0
    if imgs.ndim == 3:
        imgs = imgs[..., np.newaxis]      # Adiciona dimensão de canal: (N, H, W, 1)
    imgs   = np.repeat(imgs, 3, axis=-1)  # Replica para 3 canais: (N, H, W, 3)
    labels = ds.labels.flatten().astype('int')
    return imgs, labels

x_train, y_train = preprocess(train_ds)
x_val,   y_val   = preprocess(val_ds)
x_test,  y_test  = preprocess(test_ds)

# ══════════════════════════════════════════════════════════════════════════════
# SUBSAMPLE: 1000 IMAGENS NO TOTAL (800 treino / 100 validação / 100 teste)
# Por que subamostrar?
#   - O PneumoniaMNIST original tem ~5.800 imagens de treino.
#   - Usamos apenas 1000 para simular um cenário mais realista de dados médicos escassos
#     e para que o treino caiba confortavelmente no tempo do workshop.
#   - O SEED garante que o subconjunto seja sempre o mesmo, tornando os resultados reproduzíveis.
# ══════════════════════════════════════════════════════════════════════════════
rng_sub = np.random.default_rng(SEED)
for split, x_attr, y_attr, n in [
    ('x_train', x_train, y_train, 800),
    ('x_val',   x_val,   y_val,   100),
    ('x_test',  x_test,  y_test,  100),
]:
    idx = rng_sub.choice(len(x_attr), n, replace=False)  # Índices aleatórios sem repetição
    if split == 'x_train': x_train, y_train = x_attr[idx], y_attr[idx]
    elif split == 'x_val': x_val,   y_val   = x_attr[idx], y_attr[idx]
    else:                  x_test,  y_test  = x_attr[idx], y_attr[idx]

# ── Verificação dos splits ─────────────────────────────────────────────────────
# Imprime o tamanho de cada split e a distribuição de classes
# para confirmar que não há desequilíbrio severo entre Normal e Pneumonia.
class_names = ['Normal', 'Pneumonia']
for name, x, y in [('Treino', x_train, y_train), ('Validação', x_val, y_val), ('Teste', x_test, y_test)]:
    u, c = np.unique(y, return_counts=True)
    print(f'{name}: {x.shape} | {dict(zip([class_names[i] for i in u], c))}')

# ── Visualização: grade 4×4 de amostras do treino ─────────────────────────────
# Exibe 16 imagens aleatórias do conjunto de treino para inspeção visual.
# Títulos em verde = Normal, vermelho = Pneumonia.
rng = np.random.default_rng(SEED)
idx = rng.choice(len(x_train), 16, replace=False)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('PneumoniaMNIST — Amostras do Treino (subsample 800)', fontsize=13, fontweight='bold')
for ax, i in zip(axes.flat, idx):
    ax.imshow(x_train[i, :, :, 0], cmap='gray')  # Usa apenas o 1º canal (grayscale)
    ax.set_title(class_names[y_train[i]], color='red' if y_train[i]==1 else 'green', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()


## 🎬 Extração de Embeddings + t-SNE
### Prompt Card 2 — ▶ Execute a célula abaixo


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PASSO 1: CARREGAR O EfficientNetB0 (pré-treinado no ImageNet)
#
# EfficientNetB0 é uma CNN compacta e eficiente treinada em 1,28 milhão de
# imagens do ImageNet (fotos naturais: animais, objetos, paisagens).
# Ele aprendeu a detectar bordas, texturas e formas gerais.
#
# "Congelar" os parâmetros significa que NÃO vamos re-treinar o backbone —
# vamos usá-lo apenas como extrator fixo de características.
# Isso nos permite comparar os dois modelos de forma justa: a única diferença
# entre eles será o que aprenderam durante o pré-treinamento.
#
# Removemos a cabeça de classificação original (1000 classes do ImageNet) e
# ficamos apenas com:  features → avgpool → flatten  →  vetor de 1280 dimensões
# ══════════════════════════════════════════════════════════════════════════════
print('⏳ Baixando EfficientNetB0 pré-treinado no ImageNet (~21MB)...')
backbone = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')

# Congelar todos os parâmetros — nenhum gradiente será calculado para o backbone
for param in backbone.parameters():
    param.requires_grad = False

# Montar o extrator: apenas as camadas convolucionais + pooling global + flatten
extrator_effnet = nn.Sequential(
    backbone.features,  # Camadas convolucionais (extração de features)
    backbone.avgpool,   # Average pooling global: reduz (N, C, H, W) → (N, C, 1, 1)
    nn.Flatten()        # Flatten: (N, C, 1, 1) → (N, 1280)
).to(DEVICE)
print('✅ EfficientNetB0 carregado e congelado! Dimensão do embedding: 1280')

# ══════════════════════════════════════════════════════════════════════════════
# PASSO 2: EXTRAIR EMBEDDINGS COM EfficientNetB0
#
# Para cada imagem, passamos pelo extrator e obtemos um vetor de 1280 números.
# Esse vetor é o "embedding" — a representação compacta da imagem no espaço
# aprendido pelo EfficientNetB0 a partir do ImageNet.
# Fazemos isso para os 3 splits: treino, validação e teste.
# ══════════════════════════════════════════════════════════════════════════════
print('\n⏳ Extraindo embeddings com EfficientNetB0...')
emb_train_effnet = extrair_embeddings_cnn(extrator_effnet, x_train)
emb_val_effnet   = extrair_embeddings_cnn(extrator_effnet, x_val)
emb_test_effnet  = extrair_embeddings_cnn(extrator_effnet, x_test)
print(f'✅ EfficientNetB0 — shapes: Treino {emb_train_effnet.shape} | Val {emb_val_effnet.shape} | Teste {emb_test_effnet.shape}')

# ══════════════════════════════════════════════════════════════════════════════
# PASSO 3: CARREGAR O RAD-DINO (pré-treinado em 880K+ raios-X)
#
# RAD-DINO é um Vision Transformer (ViT) desenvolvido pela Microsoft e treinado
# com a técnica DINO (auto-supervisão) em 880.000+ raios-X de tórax.
# Ao contrário do EfficientNetB0, ele nunca viu fotos naturais —
# aprendeu diretamente a reconhecer padrões radiológicos: opacidades,
# consolidações, textura pulmonar, etc.
#
# A arquitetura ViT divide a imagem em patches (pedaços) e usa atenção
# self-attention para capturar relações globais entre regiões da imagem.
# O embedding que usamos é o CLS token — um vetor de 768 dimensões que
# o modelo produz como "resumo" de toda a imagem.
#
# O 'processor' é o pré-processador oficial do RAD-DINO:
# ele redimensiona e normaliza as imagens no padrão esperado pelo modelo.
# ══════════════════════════════════════════════════════════════════════════════
print('\n⏳ Baixando RAD-DINO da Microsoft (~87MB)...')
extrator_raddino  = AutoModel.from_pretrained('microsoft/rad-dino')
processor_raddino = AutoImageProcessor.from_pretrained('microsoft/rad-dino')

# Congelar todos os parâmetros do RAD-DINO
for param in extrator_raddino.parameters():
    param.requires_grad = False
extrator_raddino = extrator_raddino.to(DEVICE)
print('✅ RAD-DINO carregado e congelado! Dimensão do embedding: 768 (CLS token)')

# ══════════════════════════════════════════════════════════════════════════════
# PASSO 4: CONVERTER IMAGENS PARA PIL
#
# O processor do RAD-DINO espera imagens no formato PIL, não tensores numpy.
# A função numpy_to_pil (definida no Setup) faz essa conversão.
# ══════════════════════════════════════════════════════════════════════════════
print('\n⏳ Convertendo imagens para formato PIL (necessário para o RAD-DINO)...')
imgs_pil_train = numpy_to_pil(x_train)
imgs_pil_val   = numpy_to_pil(x_val)
imgs_pil_test  = numpy_to_pil(x_test)
print('✅ Conversão concluída!')

# ══════════════════════════════════════════════════════════════════════════════
# PASSO 5: EXTRAIR EMBEDDINGS COM RAD-DINO
#
# Mesmo processo do EfficientNetB0, mas agora com o RAD-DINO.
# O CLS token de 768 dimensões é o embedding de cada imagem.
# Este passo demora um pouco mais (~15s por split) pois o ViT é mais pesado.
# ══════════════════════════════════════════════════════════════════════════════
print('\n⏳ Extraindo embeddings com RAD-DINO (~15s por split)...')
emb_train_raddino = extrair_embeddings_raddino(extrator_raddino, processor_raddino, imgs_pil_train)
emb_val_raddino   = extrair_embeddings_raddino(extrator_raddino, processor_raddino, imgs_pil_val)
emb_test_raddino  = extrair_embeddings_raddino(extrator_raddino, processor_raddino, imgs_pil_test)
print(f'✅ RAD-DINO — shapes: Treino {emb_train_raddino.shape} | Val {emb_val_raddino.shape} | Teste {emb_test_raddino.shape}')

# ══════════════════════════════════════════════════════════════════════════════
# PASSO 6: t-SNE — VISUALIZAÇÃO DOS EMBEDDINGS
#
# t-SNE (t-distributed Stochastic Neighbor Embedding) é uma técnica de redução
# de dimensionalidade que projeta vetores de alta dimensão (1280 ou 768) em 2D,
# preservando a estrutura de vizinhança local.
#
# O que vamos observar:
#   - Se os pontos de Normal e Pneumonia formam grupos separados → o modelo
#     extraiu embeddings discriminativos para essa tarefa.
#   - Se os pontos estão misturados → o modelo não aprendeu features úteis
#     para distinguir as duas classes (neste dataset específico).
#
# Aplicamos o t-SNE apenas no conjunto de TESTE (100 imagens) para uma
# visualização rápida e sem viés de treino.
# ══════════════════════════════════════════════════════════════════════════════
print('\n⏳ Rodando t-SNE nos embeddings do conjunto de teste...')

tsne_effnet = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
emb_effnet_2d = tsne_effnet.fit_transform(emb_test_effnet)

tsne_raddino = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
emb_raddino_2d = tsne_raddino.fit_transform(emb_test_raddino)

print('✅ t-SNE concluído!')

# ── Plotar os resultados lado a lado ──────────────────────────────────────────
# Cada ponto = uma imagem de raio-X do conjunto de teste.
# Azul = Normal | Vermelho = Pneumonia
color_palette = {0: 'blue', 1: 'red'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Visualização t-SNE dos Embeddings de Teste', fontsize=16, fontweight='bold')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Normal',    markerfacecolor='blue', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='Pneumonia', markerfacecolor='red',  markersize=8)
]

sns.scatterplot(x=emb_effnet_2d[:, 0], y=emb_effnet_2d[:, 1],
                hue=y_test, palette=color_palette, legend=False, ax=axes[0])
axes[0].set_title('EfficientNetB0 (ImageNet)')
axes[0].set_xlabel('t-SNE Componente 1')
axes[0].set_ylabel('t-SNE Componente 2')
axes[0].legend(handles=legend_elements, title='Classe')

sns.scatterplot(x=emb_raddino_2d[:, 0], y=emb_raddino_2d[:, 1],
                hue=y_test, palette=color_palette, legend=False, ax=axes[1])
axes[1].set_title('RAD-DINO (Radiologia)')
axes[1].set_xlabel('t-SNE Componente 1')
axes[1].set_ylabel('t-SNE Componente 2')
axes[1].legend(handles=legend_elements, title='Classe')

plt.tight_layout(rect=[0, 0.03, 1, 0.9])
plt.show()


## 🏗️ Classificador 1: EfficientNetB0 (ImageNet)
### Prompt Card 1 — ✦ Selecione a célula abaixo, cole o prompt no Gemini e clique em Inserir


In [ ]:
# ✦ Prompt Card 1 — o Gemini vai inserir o código aqui


## 🏗️ Classificador 2: RAD-DINO (Radiologia)
### Prompt Card 2 — ✦ Selecione a célula abaixo, cole o prompt no Gemini e clique em Inserir


In [ ]:
# ✦ Prompt Card 2 — o Gemini vai inserir o código aqui


## 📊 Comparação de Resultados
### Prompt Card 3 — ✦ Selecione a célula abaixo, cole o prompt no Gemini e clique em Inserir


In [ ]:
# ✦ Prompt Card 3 — o Gemini vai inserir o código aqui


## 🔎 Inferência em Imagem Própria
### Prompt Card 4 — ✦ Selecione a célula abaixo, cole o prompt no Gemini e clique em Inserir


In [ ]:
# ✦ Prompt Card 4 — o Gemini vai inserir o código aqui


---
---
# 💬 Discussão Clínica

## O que aconteceu?

Ambos os backbones ficaram **completamente congelados** durante o experimento. Apenas a cabeça classificadora foi treinada — e era idêntica para os dois:

| | EfficientNetB0 | RAD-DINO |
|---|---|---|
| **Pré-treinamento** | ImageNet (1,28M fotos naturais) | 880K+ RX de tórax (MIMIC, CheXpert, NIH, PadChest, BRAX) |
| **Arquitetura** | CNN (EfficientNet) | ViT-B/14 com DINOv2 |
| **Dim. embedding** | 1.280 | 768 |
| **Parâmetros treinados** | ~262K (só head) | ~262K (só head) |
| **Tamanho do modelo** | ~21MB | ~87MB |

## Perguntas para reflexão

**1. Por que o t-SNE do RAD-DINO mostra clusters mais separados?**
> Os embeddings do RAD-DINO já codificam padrões clínicos radiológicos (opacidades, consolidações, textura pulmonar anormal). O EfficientNetB0 ImageNet codifica bordas e texturas de imagens naturais — parcialmente útil, mas sem especificidade clínica.

**2. Por que treinamos APENAS a cabeça (feature extraction)?**
> Fine-tuning completo com 5.856 imagens em um backbone de 86M+ parâmetros causaria overfitting severo. Feature extraction é a abordagem padrão quando o dataset clínico é pequeno.

**3. O RAD-DINO foi treinado em pneumonia especificamente?**
> Não. Foi treinado de forma auto-supervisionada (DINOv2) em raios-X gerais — sem labels de diagnóstico. A separação que vemos vem do conhecimento geral de radiologia de tórax aprendido de forma não-supervisionada.

**4. Isso é suficiente para uso clínico real?**
> Não. Imagens 64×64 perdem detalhes diagnósticos críticos. O dataset tem viés geográfico (crianças de Guangzhou). Sem validação prospectiva nem aprovação regulatória. Este é um experimento educacional para demonstrar o princípio da **transferência de domínio**.

**5. Qual é a implicação prática para radiologia?**
> Ao escolher um modelo pré-treinado para fine-tuning em uma tarefa radiológica, modelos treinados em dados clínicos radiológicos tendem a superar modelos de propósito geral — mesmo quando apenas a cabeça é treinada.

---

## Referências

- **PneumoniaMNIST:** Kermany DS et al., *Cell*, 2018. doi: 10.1016/j.cell.2018.02.010
- **RAD-DINO:** Pérez-García F et al., *Nature Machine Intelligence*, 2025. doi: 10.1038/s42256-024-00965-w
- **DINOv2:** Oquab M et al., *TMLR*, 2024. arXiv: 2304.07193
- **MedMNIST v2:** Yang J et al., *Scientific Data*, 2023. doi: 10.1038/s41597-022-01721-8
- **EfficientNet:** Tan M, Le QV, *ICML*, 2019. arXiv: 1905.11946
